## Now look at the GP

In [1]:
import torch
from torch import Tensor
import gpytorch
from gpytorch.likelihoods import Likelihood
from botorch.utils.probability.truncated_multivariate_normal import TruncatedMultivariateNormal
from torch.distributions import constraints
from typing import Any, Union

from linear_operator import to_linear_operator
from linear_operator.operators import (
    ConstantDiagLinearOperator,
    DiagLinearOperator,
    KroneckerProductDiagLinearOperator,
    KroneckerProductLinearOperator,
    LinearOperator,
    RootLinearOperator,
)

from gpytorch.constraints import GreaterThan, Interval
from gpytorch.distributions import base_distributions, MultitaskMultivariateNormal
from gpytorch.lazy import LazyEvaluatedKernelTensor
from gpytorch.likelihoods import _GaussianLikelihoodBase, Likelihood
from gpytorch.priors import Prior
from gpytorch.likelihoods.noise_models import FixedGaussianNoise, Noise
from gpytorch.distributions import Distribution

from matplotlib import pyplot as plt

In [2]:
torch.manual_seed(4)

In [3]:
class MultitaskGPModel(gpytorch.models.ApproximateGP):
    def __init__(self, num_tasks, num_latents, num_inducing_points):
        # Let's use a different set of inducing points for each latent function
        inducing_points = torch.rand(num_latents, num_inducing_points, 1)

        # We have to mark the CholeskyVariationalDistribution as batch
        # so that we learn a variational distribution for each task
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(
            inducing_points.size(-2), batch_shape=torch.Size([num_latents])
        )

        # We have to wrap the VariationalStrategy in a LMCVariationalStrategy
        # so that the output will be a MultitaskMultivariateNormal rather than a batch output
        variational_strategy = gpytorch.variational.LMCVariationalStrategy(
            gpytorch.variational.VariationalStrategy(
                self, inducing_points, variational_distribution, learn_inducing_locations=True
            ),
            num_tasks=num_tasks,
            num_latents=num_latents,
            latent_dim=-1
        )

        super().__init__(variational_strategy)

        # The mean and covariance modules should be marked as batch
        # so we learn a different set of hyperparameters
        self.mean_module = gpytorch.means.ConstantMean(batch_shape=torch.Size([num_latents]))
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel(batch_shape=torch.Size([num_latents])),
            batch_shape=torch.Size([num_latents])
        )

    def forward(self, x):
        # The forward function should be written as if we were dealing with each output
        # dimension in batch
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)

In [4]:
train_x = torch.linspace(0, 1, 100)[:,torch.newaxis]

train_y = torch.hstack([
    torch.sin(train_x * (2 * torch.pi)) + torch.randn(train_x.size()) * 0.2,
    torch.cos(train_x * (2 * torch.pi)) + torch.randn(train_x.size()) * 0.2,
    torch.sin(train_x * (2 * torch.pi)) + 2 * torch.cos(train_x * (2 * torch.pi)) + torch.randn(train_x.size()) * 0.2,
    -torch.cos(train_x * (2 * torch.pi)) + torch.randn(train_x.size()) * 0.2,
])

print(train_x.shape, train_y.shape)

torch.Size([100, 1]) torch.Size([100, 4])


In [5]:
train_y.min(dim=0).values, train_y.max(dim=0).values

(tensor([-1.2324, -1.4222, -2.5988, -1.6516]),
 tensor([1.3064, 1.2304, 2.4700, 1.2952]))

In [6]:
num_latents = 3
num_tasks = 4
num_inducing_points=25

model = MultitaskGPModel(num_tasks,num_latents,num_inducing_points)
base_likelihood = gpytorch.likelihoods.MultitaskGaussianLikelihood(num_tasks=num_tasks)

bounds = torch.tensor([
    [-1.3, 1.35],   # Task 1: sin
    [-1.5, 1.3],   # Task 2: cos  
    [-2.6, 2.5],   # Task 3: sin + 2*cos
    [-1.7, 1.3],   # Task 4: -cos
])
assert torch.all(bounds[:,0] <= train_y.min(dim=0).values) and torch.all(bounds[:,1] >= train_y.max(dim=0).values)

In [7]:
from gpytorch.likelihoods import Likelihood
class TruncatedMultitaskLikelihood(Likelihood):
    """
    Multitask Gaussian likelihood with box constraints per task.
    
    Args:
        num_tasks: Number of tasks
        bounds: Tensor of shape [num_tasks, 2] where bounds[:, 0] are lower bounds
                and bounds[:, 1] are upper bounds for each task
        rank: Rank of task covariance matrix (default: 0 for diagonal)
        **kwargs: Additional arguments for MultitaskGaussianLikelihood
    """
    
    def __init__(
        self,
        num_tasks: int,
        bounds: Tensor,
        rank: int = 0,
        batch_shape: torch.Size = torch.Size(),
        task_prior: Prior | None = None,
        noise_prior: Prior| None = None,
        noise_constraint: Interval| None = None,
        has_global_noise: bool = True,
        has_task_noise: bool = True,
    ) -> None:
        super(Likelihood, self).__init__()  # pyre-ignore[20]
        self.register_buffer('bounds', bounds)
        if noise_constraint is None:
            noise_constraint = GreaterThan(1e-4)

        if not has_task_noise and not has_global_noise:
            raise ValueError(
                "At least one of has_task_noise or has_global_noise must be specified. "
                "Attempting to specify a likelihood that has no noise terms."
            )

        if has_task_noise:
            if rank == 0:
                self.register_parameter(
                    name="raw_task_noises", parameter=torch.nn.Parameter(torch.zeros(*batch_shape, num_tasks))
                )
                self.register_constraint("raw_task_noises", noise_constraint)
                if noise_prior is not None:
                    self.register_prior("raw_task_noises_prior", noise_prior, lambda m: m.task_noises)
                if task_prior is not None:
                    raise RuntimeError("Cannot set a `task_prior` if rank=0")
            else:
                self.register_parameter(
                    name="task_noise_covar_factor",
                    parameter=torch.nn.Parameter(torch.randn(*batch_shape, num_tasks, rank)),
                )
                if task_prior is not None:
                    self.register_prior("MultitaskErrorCovariancePrior", task_prior, lambda m: m._eval_covar_matrix)
        self.num_tasks = num_tasks
        self.rank = rank

        if has_global_noise:
            self.register_parameter(name="raw_noise", parameter=torch.nn.Parameter(torch.zeros(*batch_shape, 1)))
            self.register_constraint("raw_noise", noise_constraint)
            if noise_prior is not None:
                self.register_prior("raw_noise_prior", noise_prior, lambda m: m.noise)

        self.has_global_noise = has_global_noise
        self.has_task_noise = has_task_noise

    @property
    def noise(self) -> Tensor | None:
        return self.raw_noise_constraint.transform(self.raw_noise)

    @noise.setter
    def noise(self, value: Union[float, Tensor]) -> None:
        self._set_noise(value)

    @property
    def task_noises(self) -> Tensor | None:
        if self.rank == 0:
            return self.raw_task_noises_constraint.transform(self.raw_task_noises)
        else:
            raise AttributeError("Cannot set diagonal task noises when covariance has ", self.rank, ">0")

    @task_noises.setter
    def task_noises(self, value: Union[float, Tensor]) -> None:
        if self.rank == 0:
            self._set_task_noises(value)
        else:
            raise AttributeError("Cannot set diagonal task noises when covariance has ", self.rank, ">0")

    def _set_noise(self, value: Union[float, Tensor]) -> None:
        self.initialize(raw_noise=self.raw_noise_constraint.inverse_transform(value))

    def _set_task_noises(self, value: Union[float, Tensor]) -> None:
        self.initialize(raw_task_noises=self.raw_task_noises_constraint.inverse_transform(value))

    @property
    def task_noise_covar(self) -> Tensor:
        if self.rank > 0:
            return self.task_noise_covar_factor.matmul(self.task_noise_covar_factor.transpose(-1, -2))
        else:
            raise AttributeError("Cannot retrieve task noises when covariance is diagonal.")

    @task_noise_covar.setter
    def task_noise_covar(self, value: Tensor) -> None:
        # internally uses a pivoted cholesky decomposition to construct a low rank
        # approximation of the covariance
        if self.rank > 0:
            with torch.no_grad():
                self.task_noise_covar_factor.data = to_linear_operator(value).pivoted_cholesky(rank=self.rank)
        else:
            raise AttributeError("Cannot set non-diagonal task noises when covariance is diagonal.")

    def _eval_covar_matrix(self) -> Tensor:
        covar_factor = self.task_noise_covar_factor
        noise = self.noise
        D = noise * torch.eye(self.num_tasks, dtype=noise.dtype, device=noise.device)  # pyre-fixme[16]
        return covar_factor.matmul(covar_factor.transpose(-1, -2)) + D

    def _shaped_noise_covar(
        self, shape: torch.Size, add_noise: bool | None = True, interleaved: bool = True, *params: Any, **kwargs: Any
    ) -> LinearOperator:
        if not self.has_task_noise:
            noise = ConstantDiagLinearOperator(self.noise, diag_shape=shape[-2] * self.num_tasks)
            return noise

        if self.rank == 0:
            task_noises = self.raw_task_noises_constraint.transform(self.raw_task_noises)
            task_var_lt = DiagLinearOperator(task_noises)
            dtype, device = task_noises.dtype, task_noises.device
            ckl_init = KroneckerProductDiagLinearOperator
        else:
            task_noise_covar_factor = self.task_noise_covar_factor
            task_var_lt = RootLinearOperator(task_noise_covar_factor)
            dtype, device = task_noise_covar_factor.dtype, task_noise_covar_factor.device
            ckl_init = KroneckerProductLinearOperator

        eye_lt = ConstantDiagLinearOperator(
            torch.ones(*shape[:-2], 1, dtype=dtype, device=device), diag_shape=shape[-2]
        )
        task_var_lt = task_var_lt.expand(*shape[:-2], *task_var_lt.matrix_shape)  # pyre-ignore[6]

        # to add the latent noise we exploit the fact that
        # I \kron D_T + \sigma^2 I_{NT} = I \kron (D_T + \sigma^2 I)
        # which allows us to move the latent noise inside the task dependent noise
        # thereby allowing exploitation of Kronecker structure in this likelihood.
        if add_noise and self.has_global_noise:
            noise = ConstantDiagLinearOperator(self.noise, diag_shape=task_var_lt.shape[-1])
            task_var_lt = task_var_lt + noise

        if interleaved:
            covar_kron_lt = ckl_init(eye_lt, task_var_lt)
        else:
            covar_kron_lt = ckl_init(task_var_lt, eye_lt)
        
        return covar_kron_lt

    def forward(self, function_samples: Tensor, *params: Any, **kwargs: Any) -> TruncatedMultivariateNormal:
        return TruncatedMultivariateNormal(function_samples, self.noise.sqrt(), self.bounds)
    
    def expected_log_prob(
        self, 
        observations: Tensor, 
        function_dist: MultitaskMultivariateNormal, 
        *args: Any, 
        **kwargs: Any
    ) -> Tensor:
        """
        Computes expected log probability under the truncated distribution.
        This is used during training with variational inference.
        """
        mu = function_dist.mean.flatten()
        covar = function_dist.covariance_matrix
        extended_bounds = torch.tile(self.bounds, (function_dist.mean.shape[0], 1))

        truncated_dist = TruncatedMultivariateNormal(loc=mu, covariance_matrix=covar, bounds=extended_bounds)
        
        # Compute log probability of observations
        log_prob = truncated_dist.log_prob(observations.flatten())
        
        return log_prob

In [8]:
# Create the likelihood
likelihood = TruncatedMultitaskLikelihood(
    num_tasks=num_tasks,
    bounds=bounds,
    rank=1,
)

In [9]:
base_mll = gpytorch.mlls.VariationalELBO(base_likelihood, model, num_data=train_y.size(0))
print(-base_mll(model(train_x), train_y))

tensor(9.1891, grad_fn=<NegBackward0>)


In [10]:
mll = gpytorch.mlls.VariationalELBO(likelihood, model, num_data=train_y.size(0))
print(-mll(model(train_x), train_y))

tensor(8377.4746, grad_fn=<NegBackward0>)


In [11]:
# Optimize
num_epochs = 50

model.train()
likelihood.train()

optimizer = torch.optim.Adam([
    {'params': model.parameters()},
    {'params': likelihood.parameters()},
], lr=0.1)

# We use more CG iterations here because the preconditioner introduced in the NeurIPS paper seems to be less
# effective for VI.
with gpytorch.settings.num_likelihood_samples(50):
    for i in range(num_epochs):
        # Within each iteration, we will go over each minibatch of data
        optimizer.zero_grad()
        output = model(train_x)
        loss = -mll(output, train_y)
        print(f'loss={loss.item()}')
        loss.backward()
        optimizer.step()

loss=8377.474609375
loss=3850.064208984375
loss=2348.70703125
loss=1903.404296875
loss=1688.701171875
loss=1478.899169921875
loss=1235.0966796875
loss=999.4286499023438


NanError: cholesky_cpu: 1875 of 1875 elements of the torch.Size([3, 25, 25]) tensor are NaN.

In [ ]:
# Set into eval mode
model.eval()
likelihood.eval()

# Initialize plots
fig, axs = plt.subplots(1, num_tasks, figsize=(4 * num_tasks, 3))

# Make predictions
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    test_x = torch.linspace(0, 1, 51)
    predictions = likelihood(model(test_x))
    mean = predictions.mean
    lower, upper = predictions.confidence_region()

for task, ax in enumerate(axs):
    # Plot training data as black stars
    ax.plot(train_x.detach().numpy(), train_y[:, task].detach().numpy(), 'k*')
    # Predictive mean as blue line
    ax.plot(test_x.numpy(), mean[:, task].numpy(), 'b')
    # Shade in confidence
    ax.fill_between(test_x.numpy(), lower[:, task].numpy(), upper[:, task].numpy(), alpha=0.5)
    ax.set_ylim([-3, 3])
    ax.legend(['Observed Data', 'Mean', 'Confidence'])
    ax.set_title(f'Task {task + 1}')

fig.tight_layout()
plt.show()